In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, mean_absolute_error, mean_squared_error
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.svm import LinearSVC
import kagglehub
import pandas as pd

In [2]:
path = kagglehub.dataset_download("sachinsharma1123/document-classification")

In [3]:
print(path)

/Users/yashchaudhary/.cache/kagglehub/datasets/sachinsharma1123/document-classification/versions/2


In [4]:
rows = []

with open("/Users/yashchaudhary/Documents/file.txt", "r", encoding="utf-8") as f:
    next(f)  # Skip the first line, which contains the record count

    for line in f:
        line = line.strip()
        if line:
            label, text = line.split(" ", 1)
            rows.append({"label": int(label), "text": text})

df = pd.DataFrame(rows)

print(df.head())

   label                                               text
0      1  champion products ch approves stock split cham...
1      2  computer terminal systems cpml completes sale ...
2      1  cobanco inc cbco year net shr cts vs dlrs net ...
3      1  am international inc am nd qtr jan oper shr lo...
4      1  brown forman inc bfd th qtr net shr one dlr vs...


In [5]:
print(df["label"].value_counts(normalize=True)*100)

label
1    51.777575
2    29.097539
6     4.612580
3     4.576117
8     3.755697
7     3.463993
4     1.969006
5     0.747493
Name: proportion, dtype: float64


In [6]:
print(df["text"].nunique())

5423


In [7]:
print(df.shape)

(5485, 2)


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5485 entries, 0 to 5484
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   5485 non-null   int64 
 1   text    5485 non-null   object
dtypes: int64(1), object(1)
memory usage: 85.8+ KB


In [9]:
empty_text = df["text"].fillna("").str.strip().eq("")
print(df.loc[empty_text, "label"].head())

Series([], Name: label, dtype: int64)


In [10]:
for random in list(range(1, 9)):
    print("\n")
    print(df[df["label"] == random] [["text", "label"]].sample(2, random_state=42))



                                                   text  label
720   pegasus gold inc pgulf rd qtr dec net shr prof...      1
4148  pda engineering pdas halts software shipments ...      1


                                                   text  label
3181  ftc eases allied signal ald unit restrictions ...      2
3921  first federal delaware agreement extended firs...      2


                                                   text  label
3761  taiwan sees its investment in u s rising sharp...      3
309   japan moves to tighten chip export curbs the m...      3


                                                   text  label
3823  brazil seamen say strike near end about half o...      4
718   blizzard closes bosphorus blizzard conditions ...      4


                                                   text  label
3552  no proof of more chernobyl grain damage diplom...      5
2102  lyng says too late for crop decoupling this ye...      5


                                           

In [11]:
seeds = [10, 21, 42, 80]

for seed in seeds:
    print("\n")
    print(df[["label", "text"]].sample(5, random_state=seed))



      label                                               text
5093      1  reckitt and colman plc pretax profits yr mln s...
1477      6  jordan petrocanada agree oil exploration proje...
1207      2  t h e fitness rfit buys livingwell unit t h e ...
1049      2  group to boost fairchild industries fen stake ...
3100      1  international pagurian has five month profit i...


      label                                               text
4583      1  j c penney declares two for one split raises q...
1274      7  u s sells mo bills at pct stop pct mo pct stop...
3072      2  swiss banker wants bond ratings share disclosu...
824       1  anglo american gold investment co ltd aagiy ye...
4267      2  hudson s bay to sell wholesale unit hudson s b...


      label                                               text
1468      2  versar vsr to buy marietta ml unit versar inc ...
33        1  ivaco inc year net shr dlrs vs dlrs net vs rev...
8         1  united presidential corp upco th qtr

In [12]:
df["article"] = df["text"].str.lower().str.strip()
df = df.drop_duplicates(subset="article").copy()

In [13]:
print("Dataset shape: ", df.shape)
print(df[["article", "text"]].head(3))

Dataset shape:  (5423, 3)
                                             article  \
0  champion products ch approves stock split cham...   
1  computer terminal systems cpml completes sale ...   
2  cobanco inc cbco year net shr cts vs dlrs net ...   

                                                text  
0  champion products ch approves stock split cham...  
1  computer terminal systems cpml completes sale ...  
2  cobanco inc cbco year net shr cts vs dlrs net ...  


In [14]:
conflicts = ( 
    df.groupby("article")["label"]
    .nunique()
    .loc[lambda labels: labels > 1]
)

print("Texts with conflicting labels:", len(conflicts))

Texts with conflicting labels: 0


In [15]:
X = df["article"]
y = df["label"]

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [17]:
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfid = vectorizer.fit_transform(X_train)
X_test_tfid = vectorizer.transform(X_test)

In [18]:
count = CountVectorizer(max_features=5000)
X_train_count = count.fit_transform(X_train)
X_test_count = count.transform(X_test)

In [20]:
model = LinearRegression()

In [21]:
model_count = LinearRegression()

In [22]:
svc_model = LinearSVC(max_iter=5000, random_state=42)

In [23]:
model.fit(X_train_tfid, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](5000,)","[ -2.41, 4.18, -8.81,..., -1.55,-11.53, -1.69]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,0.9408
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,5000


In [24]:
model_count.fit(X_train_count, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](5000,)","[ 0.19, 4.62,-0.92,...,-2.51,-2.43, 0.68]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,1.138
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,5000


In [25]:
svc_model.fit(X_train_count, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo random number generation for shuffling the data forthe dual coordinate descent (if ``dual=True``). When ``dual=False`` theunderlying implementation of :class:`LinearSVC` is not random and``random_state`` has no effect on the results.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"max_iter max_iter: int, default=1000The maximum number of iterations to be run.",5000
,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are su

In [26]:
print("For tfidfvectorizer: ")
print("Training R2: ", model.score(X_train_tfid, y_train))
print("Testing R2: ", model.score(X_test_tfid, y_test))

For tfidfvectorizer: 
Training accuracy:  0.9999316448553556
Testing accuracy:  -2.3622178848649726


In [27]:
print("For CountVectorizer: ")
print("Training R2: ", model_count.score(X_train_count, y_train))
print("Testing R2: ", model_count.score(X_test_count, y_test))

For CountVectorizer: 
Training accuracy:  0.9996478499283721
Testing accuracy:  -19.878758004284858


In [28]:
print("LinearSVC using CountVectorizer: ")
print("Training R2: ", svc_model.score(X_train_count, y_train))
print("Testing R2: ", svc_model.score(X_test_count, y_test))

LinearSVC using CountVectorizer: 
Training accuracy:  1.0
Testing accuracy:  0.967741935483871


In [29]:
y_count_pred = model_count.predict(X_test_count)
y_svc_pred = svc_model.predict(X_test_count)

In [30]:
for name, pred in [
    ("Linear regression", y_count_pred),
    ("LinearSVC", y_svc_pred)
]:
    print(f"for {name}: ") 
        
    print("MAE: ", mean_absolute_error(y_test, pred))
    print("RMSE: ", mean_squared_error(y_test, pred) ** 0.5)

for Linear regression: 
MAE:  4.617261391534349
RMSE:  8.56552761592771
for LinearSVC: 
MAE:  0.08294930875576037
RMSE:  0.5744161531944338


In [ ]:
#ConfusionMatrixDisplay.from_predictions(y_test, y_count_pred)

In [ ]:
#ConfusionMatrixDisplay.from_predictions(y_test, y_svc_pred)

In [33]:
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train_count, y_train)

,"strategy strategy: {""mean"", ""median"", ""quantile"", ""constant""}, default=""mean""Strategy to use to generate predictions.* ""mean"": always predicts the mean of the training set* ""median"": always predicts the median of the training set* ""quantile"": always predicts a specified quantile of the training set, provided with the quantile parameter.* ""constant"": always predicts a constant value that is provided by the user.",'mean'
,"constant constant: int or float or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
,"quantile quantile: float in [0.0, 1.0], default=NoneThe quantile to predict using the ""quantile"" strategy. A quantile of0.5 corresponds to the median, while 0.0 to the minimum and 1.0 to themaximum.",None
Name,Type,Value
"constant_ constant_: ndarray of shape (1, n_outputs)Mean or median or quantile of the training targets or constant valuegiven by the user.","ndarray[float64](1, 1)",[[2.14]]
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,5000
n_outputs_ n_outputs_: intNumber of outputs.,int,1


In [34]:
baseline_pred = baseline.predict(X_test_count)

In [35]:
print("Baseline R2 score: ", baseline.score(X_train_count, y_train))
print("Baseline R2 score: ", baseline.score(X_test_count, y_test))

Baseline R2 score:  0.0
Baseline R2 score:  -3.2765897550746104e-06
